# Wafer Map of Downstream Test Status

This notebook uses the simplified public interpretation of the synthetic downstream data:

- `Pass downstream spec`
- `Fail downstream spec`
- `Not tested downstream`

The generator now places dies on a **regular lattice clipped by a circular wafer footprint**.
That means the wafer maps can be drawn as real die squares without interpolation blur.

Important:

- The main regression target in the project is `lambda_res_nm`.
- The public downstream CSV contains only usable downstream test records.
- If a die is missing from the downstream CSV, we simply treat it as `Not tested downstream`.

So the public logic is now intentionally simple:

- row exists and `test_pass = 1` -> `Pass`
- row exists and `test_pass = 0` -> `Fail`
- no downstream row -> `Not tested`


In [ ]:
import sys
sys.path.append('..')

import math

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

from src.physics import MRRParameters
from src.utils import load_sources

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 8)
params = MRRParameters()

In [ ]:
df_inline, df_downstream = load_sources(input_dir='../data', prefix='synthetic')

pitch_x = np.diff(np.sort(df_inline['x_mm'].unique())).min()
pitch_y = np.diff(np.sort(df_inline['y_mm'].unique())).min()
wafer_radius = float(df_inline['r_mm'].max()) + 0.5 * max(pitch_x, pitch_y)

print(f"Inline rows: {len(df_inline)}")
print(f"Downstream tested rows: {len(df_downstream)}")
print(f"Not tested rows: {len(df_inline) - len(df_downstream)}")
print(f"All public downstream rows have test_valid = {sorted(df_downstream['test_valid'].unique())}")
print()
print('Downstream pass rule used in the generator:')
print(f"  lambda_res_nm in [{params.lambda_spec_min:.1f}, {params.lambda_spec_max:.1f}]")
print(f"  q_loaded >= {params.q_spec_min:,.0f}")
print()
print(f"Lattice pitch: dx={pitch_x:.2f} mm, dy={pitch_y:.2f} mm")
print(f"Display wafer radius: {wafer_radius:.2f} mm")

In [ ]:
df_status = df_inline.merge(
    df_downstream[['wafer_id', 'die_id', 'lambda_res_nm', 'q_loaded', 'test_pass']],
    on=['wafer_id', 'die_id'],
    how='left',
)

def classify_status(row):
    if pd.isna(row['test_pass']):
        return 'Not tested downstream'
    if row['test_pass'] == 1:
        return 'Pass downstream spec'
    return 'Fail downstream spec'

def classify_reason(row):
    if not row['tested_downstream']:
        return 'Not tested downstream'
    if row['lambda_in_spec'] and row['q_in_spec']:
        return 'Valid pass'
    if (not row['lambda_in_spec']) and (not row['q_in_spec']):
        return 'Valid fail: lambda and Q out of spec'
    if not row['lambda_in_spec']:
        return 'Valid fail: lambda out of spec'
    return 'Valid fail: Q below threshold'

df_status['tested_downstream'] = df_status['test_pass'].notna()
df_status['lambda_in_spec'] = (
    df_status['tested_downstream']
    & df_status['lambda_res_nm'].between(params.lambda_spec_min, params.lambda_spec_max)
)
df_status['q_in_spec'] = (
    df_status['tested_downstream']
    & (df_status['q_loaded'] >= params.q_spec_min)
)
df_status['downstream_status'] = df_status.apply(classify_status, axis=1)
df_status['status_reason'] = df_status.apply(classify_reason, axis=1)
df_status['downstream_status'] = pd.Categorical(
    df_status['downstream_status'],
    categories=[
        'Pass downstream spec',
        'Fail downstream spec',
        'Not tested downstream',
    ],
    ordered=True,
)

df_status[['wafer_id', 'die_id', 'downstream_status', 'status_reason']].head()

In [ ]:
print('Detailed reason breakdown behind the 3 plotted categories:')
print(df_status['status_reason'].value_counts().to_string())
print()
print('Plotted categories:')
print(
    df_status['downstream_status'].value_counts().reindex(
        ['Pass downstream spec', 'Fail downstream spec', 'Not tested downstream']
    ).to_string()
)

In [ ]:
wafer_summary = (
    df_status.groupby(['wafer_id', 'downstream_status'], observed=False)
    .size()
    .unstack(fill_value=0)
    .sort_values(['Fail downstream spec', 'Not tested downstream'], ascending=False)
)

wafer_summary.head()

## Lambda Wafer Map

This figure uses **colored die squares** for the selected wafer instead of interpolated contours.
So what you see is always tied to actual die sites, not a smoothed field between them.

- tested dies are colored by `lambda_res_nm`
- not-tested dies are grey
- the black circle is the wafer outline

In [ ]:
lambda_wafer_summary = (
    df_status[df_status['tested_downstream']]
    .groupby('wafer_id')['lambda_res_nm']
    .agg(['count', 'std', 'min', 'max'])
    .sort_values(['std', 'count'], ascending=False)
)
selected_wafer = lambda_wafer_summary.index[0]
wafer_df = df_status[df_status['wafer_id'] == selected_wafer].copy()

lambda_min = float(df_downstream['lambda_res_nm'].min())
lambda_max = float(df_downstream['lambda_res_nm'].max())
norm = mpl.colors.Normalize(vmin=lambda_min, vmax=lambda_max)
cmap = plt.cm.turbo

print(f'Selected wafer for lambda map: {selected_wafer}')
print('Chosen by highest within-wafer lambda variation among tested dies.')
print(lambda_wafer_summary.loc[selected_wafer].to_string())
print()
print(wafer_df['downstream_status'].value_counts().to_string())

fig, ax = plt.subplots(figsize=(8.8, 8.8))
for _, row in wafer_df.iterrows():
    if row['tested_downstream']:
        facecolor = cmap(norm(row['lambda_res_nm']))
    else:
        facecolor = '#D9D9D9'

    rect = patches.Rectangle(
        (row['x_mm'] - 0.5 * pitch_x, row['y_mm'] - 0.5 * pitch_y),
        pitch_x,
        pitch_y,
        facecolor=facecolor,
        edgecolor='black',
        linewidth=0.35,
    )
    ax.add_patch(rect)

wafer_outline = patches.Circle((0, 0), wafer_radius, fill=False, edgecolor='black', linewidth=1.2)
ax.add_patch(wafer_outline)

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.86, pad=0.03)
cbar.set_label('lambda_res_nm', rotation=90)

legend_handles = [
    patches.Patch(facecolor='#D9D9D9', edgecolor='black', label='Not tested downstream'),
]
ax.legend(handles=legend_handles, loc='upper right', frameon=True)
ax.set_title(f'Wafer map of lambda_res_nm by die: {selected_wafer}', fontsize=14)
ax.set_xlabel('x_mm')
ax.set_ylabel('y_mm')
axis_pad = max(pitch_x, pitch_y)
ax.set_xlim(-wafer_radius - axis_pad, wafer_radius + axis_pad)
ax.set_ylim(-wafer_radius - axis_pad, wafer_radius + axis_pad)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.18)
plt.show()

## All-Wafer Status Maps

This figure uses the same **real die squares** for every wafer, but colors them by status instead:

- blue = `Pass downstream spec`
- orange = `Fail downstream spec`
- grey = `Not tested downstream`

Because the generator now uses a circular die lattice, the wafer silhouettes should look round rather than square.

In [ ]:
palette = {
    'Pass downstream spec': '#2C7FB8',
    'Fail downstream spec': '#D95F0E',
    'Not tested downstream': '#D9D9D9',
}

wafer_ids = sorted(df_status['wafer_id'].unique())
n_cols = 5
n_rows = math.ceil(len(wafer_ids) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.8 * n_cols, 4.8 * n_rows),
    sharex=True,
    sharey=True,
)
axes = axes.flatten()

for ax, wafer_id in zip(axes, wafer_ids):
    wafer_df = df_status[df_status['wafer_id'] == wafer_id].copy()

    for _, row in wafer_df.iterrows():
        rect = patches.Rectangle(
            (row['x_mm'] - 0.5 * pitch_x, row['y_mm'] - 0.5 * pitch_y),
            pitch_x,
            pitch_y,
            facecolor=palette[row['downstream_status']],
            edgecolor='white',
            linewidth=0.35,
        )
        ax.add_patch(rect)

    counts = wafer_df['downstream_status'].value_counts()
    pass_count = int(counts.get('Pass downstream spec', 0))
    fail_count = int(counts.get('Fail downstream spec', 0))
    not_tested_count = int(counts.get('Not tested downstream', 0))

    wafer_outline = patches.Circle((0, 0), wafer_radius, fill=False, edgecolor='black', linewidth=0.9)
    ax.add_patch(wafer_outline)

    ax.set_title(
        f'{wafer_id}\nPass={pass_count}  Fail={fail_count}  Not tested={not_tested_count}',
        fontsize=10,
    )
    axis_pad = max(pitch_x, pitch_y)
    ax.set_xlim(-wafer_radius - axis_pad, wafer_radius + axis_pad)
    ax.set_ylim(-wafer_radius - axis_pad, wafer_radius + axis_pad)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

for ax in axes[len(wafer_ids):]:
    ax.set_visible(False)

legend_handles = [
    patches.Patch(facecolor=color, edgecolor='white', label=label)
    for label, color in palette.items()
]

fig.legend(
    handles=legend_handles,
    loc='lower center',
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, 0.02),
)
fig.suptitle('Wafer Maps of Downstream Test Status', fontsize=16, y=0.995)
plt.tight_layout(rect=(0, 0.06, 1, 0.97))
plt.show()